# Pattern 2: Planner -> Executor Loop (LangGraph + Ollama)

This notebook demonstrates decomposition: first create a plan, then execute each step in sequence.

In [ ]:
from typing import TypedDict, List

from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0, think=False, streaming=False)

In [ ]:
class PlanState(TypedDict):
    task: str
    plan_steps: List[str]
    current_step: int
    notes: List[str]

def planner_node(state: PlanState) -> PlanState:
    planning_prompt = f"""
Create a concise numbered plan (3 to 5 steps) for this task:
{state['task']}
Only return the numbered plan.
"""
    print(f"Planning for task: {state['task']}")
    print(f"Planning prompt:\n{planning_prompt}")
    plan_text = llm.invoke(planning_prompt).content
    raw_lines = [line.strip() for line in str(plan_text).splitlines() if line.strip()]
    steps = []
    for line in raw_lines:
        cleaned = line.lstrip('0123456789). -')
        if cleaned:
            steps.append(cleaned)
    if not steps:
        steps = ["Understand the problem", "Draft a solution", "Summarize clearly"]
    print(f"Generated plan steps: {steps}")
    state['plan_steps'] = steps
    state['current_step'] = 0
    state['notes'] = [f"Plan created with {len(steps)} steps."]
    return state

def executor_node(state: PlanState) -> PlanState:
    idx = state['current_step']
    step_text = state['plan_steps'][idx]
    exec_prompt = f"""
Task: {state['task']}
Current step ({idx + 1}/{len(state['plan_steps'])}): {step_text}
Give a short but useful completion for this step.
"""
    print(f"Executing step {idx + 1}: {step_text}")
    step_result = llm.invoke(exec_prompt).content
    state['notes'].append(f"Step {idx + 1}: {step_text}\n{step_result}")
    state['current_step'] = idx + 1
    return state

def route_after_executor(state: PlanState) -> str:
    if state['current_step'] >= len(state['plan_steps']):
        return 'finish'
    return 'execute'

In [ ]:
graph_builder = StateGraph(PlanState)
graph_builder.add_node('planner', planner_node)
graph_builder.add_node('executor', executor_node)

graph_builder.add_edge(START, 'planner')
graph_builder.add_edge('planner', 'executor')
graph_builder.add_conditional_edges('executor', route_after_executor, {
    'execute': 'executor',
    'finish': END,
})

graph = graph_builder.compile()

initial_state = {
    'task': 'Explain how to start learning agentic AI in 2 weeks with daily actions.',
    'plan_steps': [],
    'current_step': 0,
    'notes': [],
}

final_state = graph.invoke(initial_state)
final_state['plan_steps'], final_state['notes'][-1]

## Why this is agentic
The workflow adapts over time by maintaining state, routing, and iterative execution over a plan.